In [0]:
table_path = "workspace.default.cloud_app_events_banking_dlp"
df = spark.table(table_path)

In [0]:
from pyspark.sql.functions import col, substring_index, regexp_replace, when

#Get File Name and Extension 
- This Feature will be extracting file extension from file name. For DLP PDFs, CSVs, and Text files are higher risk than system files (like .exe) 
because they are the primary containers for sensitive client PII.

In [0]:
df_features = df.withColumn("file_name", regexp_replace(col("ObjectName"),r"\.[^.]+$", "")
                            ).withColumn("file_extension", substring_index(col("ObjectName"), ".", -1))
display(df_features.limit(100))


#Double Extension Detection:
- Identifies "masked" files ('data.csv.zip'). This flags attempts 
to bypass file-type filters by hiding sensitive extensions inside 
another filename.


In [0]:
extension_pattern = r"\.(pdf|docx|xlsx|doc|xls|ppt|pptx|csv|zip|7z|rar|py|sh|bat|ps1|sql|txt)"

df_features = df_features.withColumn(
    "double_extension_check", 
    when(col("file_name").rlike(extension_pattern), 1).otherwise(0)
)

  
                                  
display(df_features.limit(100))

#Clean Code Cell
with comments


In [0]:
from pyspark.sql.functions import col, lower, regexp_replace, substring_index, when

# Define extension pattern to match common file extensions
extension_pattern = r"\.(pdf|docx|xlsx|doc|xls|ppt|pptx|csv|zip|7z|rar|py|sh|bat|ps1|sql|txt)"

df_features_clean = df_features.withColumn(
    # Extracts everything after the last dot
    "file_extension", lower(substring_index(col("ObjectName"), ".", -1))
).withColumn(
    # Extracts everything BEFORE the last dot by searching for the last dot and replacing whats after with an empty string
    "file_name", regexp_replace(col("ObjectName"), r"\.[^.]+$", "")
).withColumn(
    # Check 1: Is there an extension left after removing one?
    "double_extension_check", 
    when(col("file_name").rlike(extension_pattern), 1).otherwise(0)
)
display(df_features_clean.limit(100))
   

#Tests Below

In [0]:
display(df_features.filter(col("double_extension_check") == 1))

In [0]:
display(df_features.filter(df.ObjectName.contains(".pdf.")))